<a href="https://colab.research.google.com/github/syedadilejazz/Complete_GenAI_Series_Bappy_euron/blob/main/Website_bot_using_Llama2%2CPinecone_%26_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Build Semantic Index means combining all the vectors together
#Semantic Search means qerying question embedding on vector DB and getting ranked result in return

#Install all the required libraries

In [2]:
!pip -q install -U langchain langchain-core langchain-community langchain-huggingface langchain_openai

!pip install -U langchain-chroma chromadb

!pip -q install -U bitsandbytes accelerate transformers

!pip -q install -U datasets loralib sentencepiece

!pip -q install -U pypdf

!pip -q install -U sentence-transformers

In [3]:
# !pip install openai
# !pip insall tiktoken

In [4]:
#TO consider unstructured format in website
!pip -q install unstructured

In [5]:
!pip install tokenizers

In [6]:
!pip install xformers

In [7]:
!pip install pinecone-client

#Import all the Required Libraries

In [8]:
# from langchain.document_loaders import UnstructuredURLLoader
# from langchain.text_splitter import CharacterTextSplitter
# from langchain.embedding import OpenAIEmbeddings
# from lamgchain.chat_models import ChatOpenAI
# from langchain.vectorstored import Pinecone
# import pinecone
# from langchain.chains import RetrievalQAWithSourceChain
# from langchain.embeddings import HuggingFaceEmbeddings
# from transformers import AutoTokenizer, AutoModelForCausalLM
# from langchain.llms import HuggingFacePipeline
# from transformers import pipeline
# from huggingface_hub import notebook_login
# import textwrap
# import sys
# import os
# import torch

In [9]:
# Web/document loading
from langchain_community.document_loaders import UnstructuredURLLoader

# Text splitting
from langchain_text_splitters import CharacterTextSplitter

# OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Pinecone
from langchain_pinecone import PineconeVectorStore
import pinecone

# Hugging Face
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

# Transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Hugging Face Hub
from huggingface_hub import notebook_login

# Utilities
import textwrap
import sys
import os
import torch

/tmp/ipykernel_19369/1214564660.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredURLLoader


In [10]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

#Pass the URLS and extract the data from these URLs

In [11]:
#Single or Multiple URLS can be passed here
URLs=[
    'https://blog.gopenai.com/paper-review-llama-2-open-foundation-and-fine-tuned-chat-models-23e539522acb',
    'https://www.mosaicml.com/blog/mpt-7b',
    'https://stability.ai/blog/stability-ai-launches-the-first-of-its-stablelm-suite-of-language-models',
    'https://lmsys.org/blog/2023-03-30-vicuna/'
    ]

In [12]:
loaders=UnstructuredURLLoader(urls=URLs)
data=loaders.load()

In [13]:
type(data)

list

In [14]:
len(data)

4

In [15]:
data[0]

Document(metadata={'source': 'https://blog.gopenai.com/paper-review-llama-2-open-foundation-and-fine-tuned-chat-models-23e539522acb'}, page_content='Please enable cookies.\n\nSorry, you have been blocked\n\nYou are unable to access medium.com\n\nWhy have I been blocked?\n\nThis website is using a security service to protect itself from online attacks. The action you just performed triggered the security solution. There are several actions that could trigger this block including submitting a certain word or phrase, a SQL command or malformed data.\n\nWhat can I do to resolve this?\n\nYou can email the site owner to let them know you were blocked. Please include what you were doing when this page came up and the Cloudflare Ray ID found at the bottom of this page.\n\nCloudflare Ray ID: a34c2cd768e8cd1f • Your IP: 34.125.192.214 • Performance & security by Cloudflare')

#Split the Text into Chunks

In [16]:
text_splitter=CharacterTextSplitter(separator='\n',
                                    chunk_size=1000,
                                    chunk_overlap=200)

In [17]:
text_chunks=text_splitter.split_documents(data)
print(type(text_chunks))
len(text_chunks)

<class 'list'>


61

In [18]:
text_chunks[0]

Document(metadata={'source': 'https://blog.gopenai.com/paper-review-llama-2-open-foundation-and-fine-tuned-chat-models-23e539522acb'}, page_content='Please enable cookies.\nSorry, you have been blocked\nYou are unable to access medium.com\nWhy have I been blocked?\nThis website is using a security service to protect itself from online attacks. The action you just performed triggered the security solution. There are several actions that could trigger this block including submitting a certain word or phrase, a SQL command or malformed data.\nWhat can I do to resolve this?\nYou can email the site owner to let them know you were blocked. Please include what you were doing when this page came up and the Cloudflare Ray ID found at the bottom of this page.\nCloudflare Ray ID: a34c2cd768e8cd1f • Your IP: 34.125.192.214 • Performance & security by Cloudflare')

In [19]:
text_chunks[1]

Document(metadata={'source': 'https://www.mosaicml.com/blog/mpt-7b'}, page_content='Skip to main content\nAI Research\nMay 5, 2023\nIntroducing MPT-7B: A New Standard for Open-Source, Commercially Usable LLMs\nby The Databricks AI Research Team\nIntroducing MPT-7B, the first entry in our MosaicML Foundation Series. MPT-7B is a transformer trained from scratch on 1T tokens of text and code. It is open source, available for commercial use, and matches the quality of LLaMA-7B. MPT-7B was trained on the MosaicML platform in 9.5 days with zero human intervention at a cost of ~$200k.\nLarge language models (LLMs) are changing the world, but for those outside well-resourced industry labs, it can be extremely difficult to train and deploy these models. This has led to a flurry of activity centered on open-source LLMs, such as the LLaMA series from Meta, the Pythia series from EleutherAI, the StableLM series from StabilityAI, and the OpenLLaMA model from Berkeley AI Research.')

#Downlaod the Huuging face Embeddings

In [20]:
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [22]:
query_result=embeddings.embed_query("Hello World hhhhhhhhhhhhhh")
print(len(query_result))

384


#Convert the Text Chunks into Embeddings and Create a knowledge base

In [23]:
# !pip install -U --force-reinstall \
#     opentelemetry-api \
#     opentelemetry-sdk \
#     opentelemetry-semantic-conventions

In [24]:
!pip install -U langchain-chroma chromadb

In [25]:
from langchain_chroma import Chroma

persist_directory = "db"

In [26]:
#Create vector and save into db
vectordb = Chroma.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)
#It is in Binary format, vectorhere.That is the demerit of local db. It can be overcome by using cloud DB.

In [27]:

#Now we can load the persisted database from disk, and use it as normal db.
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings
)

In [28]:
#Make a retriveal
retriever=vectordb.as_retriever()

#create a LLM Model

In [29]:
notebook_login()

In [30]:
model="meta-llama/Llama-2-7b-chat-hf"

In [31]:
tokenizer=AutoTokenizer.from_pretrained(model,use_auth_token=True)

In [32]:
model=AutoModelForCausalLM.from_pretrained(model,device_map='auto',dtype=torch.float16)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [45]:
pipe=pipeline("text-generation",model=model,tokenizer=tokenizer,max_length=2048,temperature=0.2,top_p=0.95,repetition_penalty=1.15)

In [46]:
llm=HuggingFacePipeline(pipeline=pipe)

In [47]:
llm.invoke("Please provide a concise summary of the Book Harry Potter")

"Please provide a concise summary of the Book Harry Potterand the Philosopher's Stone by J.K. Rowling\nHarry Potter is an orphan who lives with his cruel relatives, the Dursleys, but on his eleventh birthday he discovers that he is a wizard and begins attending Hogwarts School of Witchcraft and Wizardry. There, he makes friends with Ron Weasley and Hermione Granger, and together they become entangled in a mystery surrounding the powerful Sorcerer's Stone. They must prevent the stone from falling into the wrong hands and uncover the truth about the school's enigmatic headmaster, Professor Quirrell. Along the way, they encounter various obstacles, including the evil wizard Lord Voldemort, who murdered Harry's parents and seeks to return to power. Through their adventures, Harry learns important lessons about friendship, courage, and the importance of staying true to oneself."

#Intialize the Retrieval QA with Source Chain

In [48]:
from langchain.chains import RetrievalQA

ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
query="How good is Vicuna?"

In [ ]:
docs=vectorstore.similarity_search(query,k=3)
docs

In [ ]:
qa=RetreievalQAWithSourcesChain.from_chain_type(llm=llm,
                                                chain_type="stuff",
                                                retriever=retriever,
                                                return_source_documents=True)

In [51]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the following context.

Context:
{context}

Question:
{question}

Answer:
""")

retriever = vectordb.as_retriever(search_kwargs={"k": 4})

rag_chain = (
    {
        "context": retriever,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("How good is Vicuna?")

print(answer)

Human: 
Answer the question using only the following context.

Context:
[Document(id='48036c78-702e-49a0-a3ab-add84ae9d9c2', metadata={'source': 'https://lmsys.org/blog/2023-03-30-vicuna/'}, page_content='‹ Back to Blog\n‹ Back to Blog\nVicuna: An Open-Source Chatbot Impressing GPT-4 with 90%* ChatGPT Quality\nThe Vicuna TeamMarch 30, 2023\nWe introduce Vicuna-13B, an open-source chatbot trained by fine-tuning LLaMA on user-shared conversations collected from ShareGPT. Preliminary evaluation using GPT-4 as a judge shows Vicuna-13B achieves more than 90%* quality of OpenAI ChatGPT and Google Bard while outperforming other models like LLaMA and Stanford Alpaca in more than 90%* of cases. The cost of training Vicuna-13B is around $300. The code and weights, along with an online demo, are publicly available for non-commercial use.\nVicuna (generated by stable diffusion 2.1)\nAccording to a fun and non-scientific evaluation with GPT-4. Further rigorous evaluation is needed.\nHow Good is Vic